## Executive Summary

**Purpose:** Estimate power dissipation in motor cores

**What it does:** Calculate core (iron) losses in magnetic materials.

### Why It Matters
Thermal analysis needs core loss to calculate motor temperature rise and cooling requirements.

### Quick Start
**Inputs:** Flux density, frequency, material (steel grade or fitted parameters)

**Outputs:** Core loss in W/kg, decomposed into hysteresis and eddy current components

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Used by thermal and efficiency analysis modules

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import numpy as np
from scipy.optimize import curve_fit

## Theory: Core Loss Decomposition

Core loss arises from three mechanisms:
1. **Hysteresis loss** — Domain wall friction
2. **Eddy current loss** — Induced currents
3. **Excess loss** — Domain wall dynamics

### Steinmetz Model

$$P = k \cdot f^{\\alpha} \cdot \\hat{B}^{\\beta}$$

Simple two-exponent fit for sinusoidal excitation.

### Modified Steinmetz (iGSE)

$$P = k \cdot f^{\\alpha-1} \cdot (f \cdot \\hat{B})^{\\beta}$$

Improved for PWM drives and non-sinusoidal waveforms.

### Bertotti Three-Term

$$P_{total} = k_h f \\hat{B}^{\\beta} + k_e (f\\hat{B})^2 + k_a (f\\hat{B})^{1.5}$$

Physically-based separation into components.

In [2]:
#| export
def steinmetz(
    f: float | np.ndarray,
    B_peak: float | np.ndarray,
    k: float,
    alpha: float,
    beta: float,
) -> float | np.ndarray:
    r"""Classical Steinmetz core loss model.
    P = k * f^alpha * B^beta
    Args:
        f: Frequency (Hz)
        B_peak: Peak flux density (T)
        k: Material constant
        alpha: Frequency exponent (1.0–1.6)
        beta: Flux density exponent (1.6–2.2)
    Returns:
        Core loss (W/kg)
    """
    return k * (np.asarray(f) ** alpha) * (np.asarray(B_peak) ** beta)

In [3]:
#| export
def modified_steinmetz(
    f: float | np.ndarray,
    B_peak: float | np.ndarray,
    k: float,
    alpha: float,
    beta: float,
) -> float | np.ndarray:
    r"""Modified Steinmetz (iGSE) for PWM waveforms.
    P = k * f^(alpha-1) * (f*B)^beta
    Args:
        f: Frequency (Hz)
        B_peak: Peak flux density (T)
        k: Material constant
        alpha: Frequency exponent
        beta: Combined exponent
    Returns:
        Core loss (W/kg)
    """
    f = np.asarray(f)
    B = np.asarray(B_peak)
    return k * (f ** (alpha - 1)) * ((f * B) ** beta)

In [4]:
#| export
def bertotti(
    f: float | np.ndarray,
    B_peak: float | np.ndarray,
    k_h: float,
    k_e: float,
    k_a: float,
    alpha: float = 1.0,
    beta: float = 2.0,
) -> dict[str, float | np.ndarray]:
    r"""Bertotti three-term iron loss separation.
    Separates into hysteresis, eddy, and excess components.
    Args:
        f: Frequency (Hz)
        B_peak: Peak flux density (T)
        k_h: Hysteresis coefficient
        k_e: Eddy current coefficient
        k_a: Excess coefficient
        alpha: Frequency exponent (default 1.0)
        beta: Flux density exponent (default 2.0)
    Returns:
        Dict with 'hysteresis', 'eddy', 'excess', 'total' (W/kg)
    """
    f = np.asarray(f)
    B = np.asarray(B_peak)
    p_hyst = k_h * (f ** alpha) * (B ** beta)
    p_eddy = k_e * (f * B) ** 2
    p_exc  = k_a * (f * B) ** 1.5
    return {
        "hysteresis": p_hyst,
        "eddy": p_eddy,
        "excess": p_exc,
        "total": p_hyst + p_eddy + p_exc,
    }

In [5]:
#| export
def fit_steinmetz(
    f_arr: np.ndarray,
    B_arr: np.ndarray,
    loss_arr: np.ndarray,
) -> dict:
    """Fit Steinmetz parameters (k, α, β) to measured data."""
    def _func(X, k, alpha, beta):
        return steinmetz(X[0], X[1], k, alpha, beta)
    popt, _ = curve_fit(_func, (f_arr, B_arr), loss_arr,
                        p0=[0.01, 1.5, 2.5], maxfev=10000)
    fitted = _func((f_arr, B_arr), *popt)
    ss_res = np.sum((loss_arr - fitted) ** 2)
    r2 = 1.0 - ss_res / np.sum((loss_arr - np.mean(loss_arr)) ** 2)
    rmse = float(np.sqrt(np.mean((loss_arr - fitted) ** 2)))
    return {"k": popt[0], "alpha": popt[1], "beta": popt[2],
            "r2": float(r2), "rmse": rmse, "model": "Steinmetz"}

## Tests

Iron loss models and coefficient fitting validated against synthetic data.

In [ ]:
#| hide
import numpy as np
from emachines.magnetics.iron_loss import (
    MODEL_NAMES, bertotti, fit_bertotti, fit_loss_model,
    fit_modified_steinmetz, fit_steinmetz, modified_steinmetz, steinmetz)

# ── forward models ────────────────────────────────────────────────────
assert steinmetz(100, 1.0, 0.01, 1.5, 2.0) > steinmetz(50, 1.0, 0.01, 1.5, 2.0)
assert steinmetz(50, 1.5, 0.01, 1.5, 2.0) > steinmetz(50, 1.0, 0.01, 1.5, 2.0)
assert modified_steinmetz(400, 1.5, 0.01, 1.5, 2.0) > 0
assert modified_steinmetz(400, 1.0, 0.01, 1.5, 2.0) > modified_steinmetz(50, 1.0, 0.01, 1.5, 2.0)

r = bertotti(50, 1.5, k_h=0.02, k_e=1e-4, k_a=1e-3)
assert np.isclose(r["total"], r["hysteresis"] + r["eddy"] + r["excess"])
r2 = bertotti(400, 1.0, k_h=0.01, k_e=5e-5, k_a=5e-4)
assert r2["hysteresis"] > 0 and r2["eddy"] > 0 and r2["excess"] > 0
print("✓ bertotti / steinmetz / modified_steinmetz")

# ── fitting: synthetic data from known Bertotti coefficients ──────────
k_h, k_e, k_a = 0.02, 1e-4, 1e-3
freqs  = np.array([50,50,50,100,100,100,400,400,400], dtype=float)
B_vals = np.array([0.5,1.0,1.5,0.5,1.0,1.5,0.5,1.0,1.5], dtype=float)
loss   = bertotti(freqs, B_vals, k_h, k_e, k_a)["total"]

res = fit_bertotti(freqs, B_vals, loss)
assert np.isclose(res["k_h"], k_h, rtol=1e-3), f"k_h={res['k_h']}"
assert np.isclose(res["k_e"], k_e, rtol=1e-3), f"k_e={res['k_e']}"
assert np.isclose(res["k_a"], k_a, rtol=1e-3), f"k_a={res['k_a']}"
assert res["r2"] > 0.999 and res["model"] == "Bertotti"
print("✓ fit_bertotti")

res_s = fit_steinmetz(freqs, B_vals, loss)
assert "k" in res_s and "alpha" in res_s and "beta" in res_s
assert 0 < res_s["r2"] <= 1.0 and res_s["model"] == "Steinmetz"

res_ms = fit_modified_steinmetz(freqs, B_vals, loss)
assert res_ms["model"] == "Modified Steinmetz"
print("✓ fit_steinmetz / fit_modified_steinmetz")

for model in MODEL_NAMES:
    r = fit_loss_model(freqs, B_vals, loss, model=model)
    assert r["model"] == model and r["r2"] > 0.0

try:
    fit_loss_model(freqs, B_vals, loss, model="SomeUnknownModel")
    assert False
except ValueError:
    pass
print("✓ fit_loss_model dispatch / unknown-model error")
print("\n✓ All iron loss tests passed")
